In [27]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

import sys
sys.path.append("../src")

In [28]:
from data import ChatSFTDataset, SFTCollator
from data import TASK_PREFIXES, SYSTEM_PROMPT

In [29]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen3.5-0.8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


path = "../data/llm-cl-500/FOMC/train.json"
dataset = load_dataset("json", split="train", data_files=path)

# def split_answer(example):
#     answer, last = example["answer"].split("\n", 1)
#     return {"answer": answer, "reasoning": last}

# dataset = dataset.map(split_answer)

dataset

Dataset({
    features: ['prompt', 'answer'],
    num_rows: 500
})

In [30]:
train_dataset = ChatSFTDataset(
    dataset=dataset,
    tokenizer=tokenizer,
    task_prefixes=TASK_PREFIXES,
    system_prompt=SYSTEM_PROMPT,
    task_name="scienceqa",   # useful if the JSONL does not have a "task" column
    max_length=512,
    enable_thinking=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=SFTCollator(tokenizer),
)

In [31]:
def print_example(dataset, tokenizer, idx: int = 0):
    item = dataset[idx]

    print("=" * 80)
    print("RAW ITEM")
    print("=" * 80)
    print(item)

    input_ids = item["input_ids"]
    labels = item["labels"]
    attention_mask = item["attention_mask"]

    print("\n" + "=" * 80)
    print("SHAPES")
    print("=" * 80)
    print("input_ids:", input_ids.shape)
    print("labels:", labels.shape)
    print("attention_mask:", attention_mask.shape)

    assert input_ids.shape == labels.shape
    assert input_ids.shape == attention_mask.shape

    print("\n" + "=" * 80)
    print("DECODED FULL INPUT")
    print("=" * 80)
    print(tokenizer.decode(input_ids, skip_special_tokens=False))

    supervised_token_ids = [
        token_id.item()
        for token_id, label in zip(input_ids, labels)
        if label.item() != -100
    ]

    print("\n" + "=" * 80)
    print("DECODED SUPERVISED TARGET ONLY")
    print("=" * 80)
    print(tokenizer.decode(supervised_token_ids, skip_special_tokens=False))

    num_supervised_tokens = sum(label.item() != -100 for label in labels)
    num_ignored_tokens = sum(label.item() == -100 for label in labels)

    print("\n" + "=" * 80)
    print("LOSS MASK")
    print("=" * 80)
    print("ignored prompt tokens:", num_ignored_tokens)
    print("supervised answer tokens:", num_supervised_tokens)

    assert num_supervised_tokens > 0, "No supervised answer tokens found."
    assert num_ignored_tokens > 0, "Prompt tokens are not masked."

In [32]:
print_example(train_dataset, tokenizer, idx=0)

RAW ITEM
{'input_ids': tensor([248045,   8678,    198,   2523,    513,    264,  61446,  17313,     13,
         21134,    279,   3274,   6681,    430,  10897,     13,   3301,   1132,
           279,   1534,   4087,     13, 248046,    198, 248045,    846,    198,
         15666,    279,   5081,  60514,   3296,     13,   3301,   1132,    279,
          4252,   2904,     13,    271,   3710,    369,    279,  31219,   4687,
         27989,    364,    279,   2614,   1414,     30,    357,     13,  59201,
           786,     11,    417,     13,  73229,    786,     11,    351,     13,
         20002,     13,  21513,    799,    494,    357,     11,    417,    321,
           351,     13,    198,   1138,     25,    198,  10883,     11,    975,
         12653,  12784,    421,  10633,    314,   4860,   9375,  23752,  19252,
         13849,    494,   5713,  22591,    995,  14195,  14733,    314,   3200,
            11,    321,    421,  10258,   5792,  10633,    314,   4860,   9375,
         23752,  

In [33]:
item = train_dataset[0]

print(tokenizer.decode(item["input_ids"], skip_special_tokens=False))

target_ids = item["input_ids"][item["labels"] != -100]
print("TARGET:")
print(tokenizer.decode(target_ids, skip_special_tokens=False))

<|im_start|>system
You are a concise assistant. Answer the task exactly as requested. Return only the final answer.<|im_end|>
<|im_start|>user
Answer the multiple-choice question. Return only the correct option.

What is the monetary policy stance for the following text? A. dovish, B. hawkish, C. neutral. Choose one from A, B and C.
Text:
However, other participants observed that measures of longer-term inflation compensation derived from financial instruments had remained stable of late, and that survey-based measures of longer-term inflation expectations also had not changed appreciably, on net, in recent months.
Stance:<|im_end|>
<|im_start|>assistant
<think>

</think>

C<|im_end|>

TARGET:
C<|im_end|>



In [35]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["labels"].shape)
print(batch["attention_mask"].shape)

print(tokenizer.decode(batch["input_ids"][0], skip_special_tokens=False))

target_ids = batch["input_ids"][0][batch["labels"][0] != -100]
print("TARGET:")
print(tokenizer.decode(target_ids, skip_special_tokens=False))

torch.Size([2, 141])
torch.Size([2, 141])
torch.Size([2, 141])
<|im_start|>system
You are a concise assistant. Answer the task exactly as requested. Return only the final answer.<|im_end|>
<|im_start|>user
Answer the multiple-choice question. Return only the correct option.

What is the monetary policy stance for the following text? A. dovish, B. hawkish, C. neutral. Choose one from A, B and C.
Text:
However, other participants observed that measures of longer-term inflation compensation derived from financial instruments had remained stable of late, and that survey-based measures of longer-term inflation expectations also had not changed appreciably, on net, in recent months.
Stance:<|im_end|>
<|im_start|>assistant
<think>

</think>

C<|im_end|>

TARGET:
C<|im_end|>

